## Networks, volumes & environment

Three service-shaping blocks, each carrying a module you already know into Compose.

**Networks.** You can go years on the auto-created default network, but explicit networks buy **tier separation** — a reverse proxy that can see the API, an API that can see the DB, but a proxy that *can't* reach the DB directly (the segmentation-by-attachment idea from module 05):

```yaml
services:
  proxy: { networks: [frontend] }
  api:   { networks: [frontend, backend] }
  db:    { networks: [backend] }
networks:
  frontend: {}
  backend: {}
```

An **external network** (`external: true`) connects to one created outside this file — by another project or `docker network create`.

**Volumes** carry module 04 over: declare named volumes at the top level, reference them in services. The Compose bonus is lifecycle tied to the project — `down -v` removes them, plain `down` keeps them. And **Compose resolves bind-mount relative paths against the `compose.yaml` directory**, so `./src:/app/src` works no matter where you ran `up`.

**Environment** has **three layers**, lowest precedence to highest: (1) the image's `ENV`, (2) **`env_file:`** (files of `KEY=value`), (3) **`environment:`** in the YAML, which wins.

```yaml
api:
  env_file: [./api.env]
  environment:
    LOG_LEVEL: debug        # overrides api.env
```

**The `.env` gotcha.** Compose *also* reads a file literally named **`.env`**, but for a totally different job: **substituting `${VAR}` into the YAML text itself** (`image: myorg/api:${TAG}`), resolved at parse time. So `.env` fills in the *YAML*; `env_file:` fills in the *container's* environment. They're constantly confused — read the keyword, not the dots. Substitutions also support defaults and required values: `${TAG:-latest}` (default) and `${DB_PASSWORD:?must be set}` (error if missing).